This notebook is to extract movies data from a kaggle dataset : https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata

In [1]:
import numpy as np
import pandas as pd

In [2]:
movies = pd.read_csv("../data/tmdb_5000_movies.csv")
credit = pd.read_csv("../data/tmdb_5000_credits.csv")

## Extract movies information for our movies table

In [3]:
movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [4]:
movies_df = movies[["title", "release_date"]]
movies_df.head()

,title,release_date
0,Avatar,2009-12-10
1,Pirates of the Caribbean: At World's End,2007-05-19
2,Spectre,2015-10-26
3,The Dark Knight Rises,2012-07-16
4,John Carter,2012-03-07


## Extract information about Genres (Genre table) and Movies genre details (Movie_genres table)

In [5]:
print(movies['genres'].head())

0    [{"id": 28, "name": "Action"}, {"id": 12, "nam...
1    [{"id": 12, "name": "Adventure"}, {"id": 14, "...
2    [{"id": 28, "name": "Action"}, {"id": 12, "nam...
3    [{"id": 28, "name": "Action"}, {"id": 80, "nam...
4    [{"id": 28, "name": "Action"}, {"id": 12, "nam...
Name: genres, dtype: object


As we can see. A genre is defined my the name and the id

In [6]:
from ast import literal_eval

In [7]:
movies["genres"] = movies["genres"].apply(literal_eval)

In [8]:
def get_list(x):
    if isinstance(x,list):
        names = [i['name'] for i in x]
        if len(names) > 3:
            names=names[:3]
        return names
    return []

In [9]:
movies["genres"] = movies["genres"].apply(get_list)

In [10]:
genres_df = pd.DataFrame({"name":pd.unique(movies['genres'].explode())})
genres_df

,name
0,Action
1,Adventure
2,Fantasy
3,Crime
4,Drama
5,Science Fiction
6,Animation
7,Family
8,Thriller
9,Western


In [11]:
genres_df = genres_df.fillna('Unknown')

In [12]:
genres_df

,name
0,Action
1,Adventure
2,Fantasy
3,Crime
4,Drama
5,Science Fiction
6,Animation
7,Family
8,Thriller
9,Western


In [13]:
movies_genre_df = movies[['title', 'genres']].explode('genres')
movies_genre_df

,title,genres
0,Avatar,Action
0,Avatar,Adventure
0,Avatar,Fantasy
1,Pirates of the Caribbean: At World's End,Adventure
1,Pirates of the Caribbean: At World's End,Fantasy
...,...,...
4800,"Signed, Sealed, Delivered",Comedy
4800,"Signed, Sealed, Delivered",Drama
4800,"Signed, Sealed, Delivered",Romance
4801,Shanghai Calling,NaN


In [14]:
def extract_index(x):
  if pd.isna(x):
    return np.nan
  matched_rows_indices = genres_df[genres_df['name'] == x].index
  if not matched_rows_indices.empty:
    return int(matched_rows_indices[0])
  else:
    return np.nan

In [15]:
movies_genre_df['genre_id'] = movies_genre_df['genres'].apply(extract_index)

In [16]:
movies_genre_df['genre_id'] = movies_genre_df['genre_id'].astype('Int64')
movies_genre_df

,title,genres,genre_id
0,Avatar,Action,0
0,Avatar,Adventure,1
0,Avatar,Fantasy,2
1,Pirates of the Caribbean: At World's End,Adventure,1
1,Pirates of the Caribbean: At World's End,Fantasy,2
...,...,...,...
4800,"Signed, Sealed, Delivered",Comedy,10
4800,"Signed, Sealed, Delivered",Drama,4
4800,"Signed, Sealed, Delivered",Romance,11
4801,Shanghai Calling,NaN,<NA>


## Extract information about actors and Movies actors details

In [17]:
credit.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [18]:
credit['cast'] = credit['cast'].apply(literal_eval)

In [19]:
credit["cast"] = credit["cast"].apply(get_list)

In [20]:
credit["cast"].head()

0    [Sam Worthington, Zoe Saldana, Sigourney Weaver]
1       [Johnny Depp, Orlando Bloom, Keira Knightley]
2        [Daniel Craig, Christoph Waltz, Léa Seydoux]
3        [Christian Bale, Michael Caine, Gary Oldman]
4      [Taylor Kitsch, Lynn Collins, Samantha Morton]
Name: cast, dtype: object

In [21]:
import random
import time
    
def str_time_prop(start, end, time_format):
    """Get a time at a proportion of a range of two formatted times.

    start and end should be strings specifying times formatted in the
    given format (strftime-style), giving an interval [start, end].
    prop specifies how a proportion of the interval to be taken after
    start.  The returned time will be in the specified format.
    """
    prop = random.random()
    stime = time.mktime(time.strptime(start, time_format))
    etime = time.mktime(time.strptime(end, time_format))

    ptime = stime + prop * (etime - stime)

    return time.strftime(time_format, time.localtime(ptime))


def random_date(start, end):
    return str_time_prop(start, end, '%Y-%m-%d')
    
print(random_date("2008-1-1", "2026-5-1"))

2013-01-19


In [22]:
unique_actors = pd.unique(credit['cast'].explode())
actors_df = pd.DataFrame({"full_name": unique_actors, "date_of_birth": [random_date("1965-1-1", "2015-5-1") for _ in range(len(unique_actors))]})

In [23]:
actors_df.head()

,full_name,date_of_birth
0,Sam Worthington,1967-01-05
1,Zoe Saldana,1992-09-02
2,Sigourney Weaver,1986-06-07
3,Johnny Depp,2008-07-23
4,Orlando Bloom,1974-11-28


In [24]:
movies_actors_df = credit[['title', 'cast']].explode('cast')

In [25]:
movies_actors_df.head()

,title,cast
0,Avatar,Sam Worthington
0,Avatar,Zoe Saldana
0,Avatar,Sigourney Weaver
1,Pirates of the Caribbean: At World's End,Johnny Depp
1,Pirates of the Caribbean: At World's End,Orlando Bloom


In [26]:
def extract_index(x):
  if pd.isna(x):
    return np.nan
  matched_rows_indices = actors_df[actors_df['full_name'] == x].index
  if not matched_rows_indices.empty:
    return int(matched_rows_indices[0])
  else:
    return np.nan

In [27]:
movies_actors_df["actor_id"] = movies_actors_df['cast'].apply(extract_index).astype('Int64')

In [28]:
movies_actors_df.head()

,title,cast,actor_id
0,Avatar,Sam Worthington,0
0,Avatar,Zoe Saldana,1
0,Avatar,Sigourney Weaver,2
1,Pirates of the Caribbean: At World's End,Johnny Depp,3
1,Pirates of the Caribbean: At World's End,Orlando Bloom,4


## Save the results

In [29]:
genres_df = genres_df.reset_index(names="id")

In [30]:
genres_df.to_csv("../data/genres.csv", index=False)

In [31]:
actors_df = actors_df.reset_index(names="id")

In [32]:
actors_df.to_csv("../data/actors.csv", index=False)

In [33]:
movies_df = movies_df.reset_index(names="id")

In [34]:
movies_df.to_csv("../data/movies.csv", index=False)

In [35]:
movies_actors_df = movies_actors_df.reset_index(names="movie_id").drop(["title","cast"],axis=1)

In [36]:
movies_actors_df.to_csv("../data/movies_actors.csv", index=False)

In [37]:
movies_genre_df = movies_genre_df.reset_index(names="movie_id").drop(["title","genres"],axis=1)

In [38]:
movies_genre_df.to_csv("../data/movies_genres.csv", index=False)

## Clean userdata randomly generated (https://www.mockaroo.com/)

In [39]:
users_df = pd.read_csv("../data/users.csv")
users_df.head()

,username,lastname,firstname,date_of_birth,password
0,cniesel0,Niesel,Clare,2003-07-17,ab43c8f7837b6a6b36a5865ca1ce90b115440cb9e6d039...
1,tdundon1,Dundon,Tove,2017-02-27,8956c78a236814270da5b12953d3f447bcfd2be33ce661...
2,tpaulino2,Paulino,Theresina,2019-03-15,8b2770247e0b094e0bf406916ea1f25c7c130cc75b7e46...
3,lbudibent3,Budibent,Liuka,1978-04-01,2bde63064bad7a41f33385b1bb0c7ef2f4d79593e8e6bf...
4,dpiatto4,Piatto,Devondra,2004-12-07,7fc6e3ae97a13db36357507c08d9efcc77cf3d0df4011b...


In [40]:
users_df = pd.read_csv("../data/users.csv")
users_df.tail()

,username,lastname,firstname,date_of_birth,password
96,wleatt2o,Leatt,Wynny,1982-12-18,59e79c47412d693f0ea1b851604f2e96e4a06644dc3e82...
97,egabbatiss2p,Gabbatiss,Evvy,1997-11-15,97076e98b63a9310c42c53d0a343e98f2d582bb96e5432...
98,mdampier2q,Dampier,My,1973-04-20,fa3039289c12522e2aa8f1ad421e190522315c39c3372a...
99,pcolleford2r,Colleford,Percy,1994-08-04,178547dd3fbcf8662b91caf7f9607b37d72e6f34efb620...
100,mmeka,Meka,Moise,2025-03-17,4661499d8129f5fb56a295ac0cdf49e9dfe49e8610b09b...


In [41]:
user_idx = np.random.choice(np.arange(len(users_df)), size=10000, replace=True)

In [42]:
user_idx[:10]

array([73, 51,  1, 33, 48, 11, 33, 63, 90, 29])

In [43]:
username_list = list(users_df['username'].iloc[user_idx])

In [44]:
username_list[:10]

['ccrippin21',
 'bdundin1f',
 'tdundon1',
 'jmccarterx',
 'astoate1c',
 'ssparshettb',
 'jmccarterx',
 'wdjordjevic1r',
 'hpoundford2i',
 'rbottrellt']

In [45]:
movie_idx = np.random.choice(np.arange(len(movies_df)), size=10000, replace=True)
movie_idx[:10]

array([ 968, 4013, 3225,  725, 4496, 2702,  486, 1807, 1606, 4507])

In [46]:
watch_date_list = [random_date("2008-1-1", "2026-5-1") for _ in range(10000)]

In [47]:
watch_date_list[:10]

['2024-10-18',
 '2015-06-07',
 '2020-03-31',
 '2008-08-21',
 '2022-12-07',
 '2014-04-06',
 '2021-08-16',
 '2011-08-11',
 '2018-04-12',
 '2023-12-30']

In [48]:
watch_movies_df = pd.DataFrame({'username': username_list,
                                'movie_id': movie_idx,
                                'watch_date': watch_date_list,
                                'rating': np.random.randint(1, 5, len(username_list)),
                                'comment': [pd.NA] * len(username_list)})

In [49]:
watch_movies_df.head()

,username,movie_id,watch_date,rating,comment
0,ccrippin21,968,2024-10-18,2,<NA>
1,bdundin1f,4013,2015-06-07,1,<NA>
2,tdundon1,3225,2020-03-31,4,<NA>
3,jmccarterx,725,2008-08-21,4,<NA>
4,astoate1c,4496,2022-12-07,2,<NA>


In [50]:
watch_movies_df.loc[watch_movies_df['username']=='mmeka']

,username,movie_id,watch_date,rating,comment
333,mmeka,2771,2014-12-19,2,NaN
620,mmeka,2113,2009-10-01,2,NaN
660,mmeka,3671,2022-10-06,1,NaN
1003,mmeka,601,2019-09-02,3,NaN
1188,mmeka,3227,2023-10-22,2,NaN
...,...,...,...,...,...
9382,mmeka,4477,2010-09-23,3,NaN
9479,mmeka,639,2015-12-01,3,NaN
9491,mmeka,1028,2009-03-11,4,NaN
9666,mmeka,549,2019-08-15,4,NaN


In [51]:
watch_movies_df.to_csv("../data/watch_movies.csv", index=False)

In [52]:
def watch_user_genres_mat(watch_movies_df:pd.DataFrame, movies_genres_df:pd.DataFrame):
    tmp = pd.merge(watch_movies_df, movies_genres_df, 
                   on="movie_id", how='left').drop(['comment', 'watch_date', 'movie_id'], axis=1)

    return tmp.groupby(['username', 'genre_id']).agg('mean').fillna(0).reset_index()

In [53]:
mat = watch_user_genres_mat(watch_movies_df, movies_genre_df)

In [54]:
mat.loc[mat['username']=='mmeka']

,username,genre_id,rating
1176,mmeka,0,2.318182
1177,mmeka,1,2.285714
1178,mmeka,2,2.333333
1179,mmeka,3,2.357143
1180,mmeka,4,2.595238
1181,mmeka,5,2.666667
1182,mmeka,6,2.500000
1183,mmeka,7,2.714286
1184,mmeka,8,2.562500
1185,mmeka,10,2.594595


In [55]:
from surprise import SVD, Dataset, Reader

In [56]:
reader = Reader()

data = Dataset.load_from_df(mat, reader)

In [57]:
train_set = data.build_full_trainset()

In [58]:
svd = SVD()

In [59]:
svd.fit(train_set)

In [60]:
svd.predict(uid='mmcj', iid='9').est

2.4845018715643756

In [61]:
import numpy as np

In [62]:
s = np.random.randn(10)
sorted(range(len(s)), key=lambda k: s[k])

[3, 0, 1, 4, 7, 2, 9, 5, 6, 8]

In [63]:
np.sort(s)

array([-2.61087034, -1.35240625, -0.71478445, -0.47437579, -0.42675292,
       -0.30292275, -0.059956  ,  0.98303663,  1.03862395,  1.33444017])

In [64]:
x = sorted(range(len(s)), key=lambda k: s[k])
s[x[:100]]

array([-2.61087034, -1.35240625, -0.71478445, -0.47437579, -0.42675292,
       -0.30292275, -0.059956  ,  0.98303663,  1.03862395,  1.33444017])

In [65]:
user_idx = np.random.choice(np.arange(len(users_df)), size=1000, replace=True)
username_list = list(users_df['username'].iloc[user_idx])

In [66]:
actors_idx = np.random.choice(np.arange(len(actors_df)), size=1000, replace=True)
genres_idx = np.random.choice(np.arange(len(genres_df)), size=1000, replace=True)

In [67]:
user_actors_df = pd.DataFrame({
'username':username_list,
'actor_id':actors_idx,
'rating': np.random.randint(1, 5, len(username_list))}) 

user_genres_df = pd.DataFrame({
'username':username_list,
'genre_id':genres_idx,
'rating': np.random.randint(1, 5, len(username_list))}) 

In [68]:
user_actors_df.head()

,username,actor_id,rating
0,ppointon2l,672,2
1,pvicaryn,522,4
2,wdjordjevic1r,453,3
3,sbignell19,5069,3
4,ppointon2l,4659,1


In [69]:
user_genres_df.head()

,username,genre_id,rating
0,ppointon2l,6,3
1,pvicaryn,3,4
2,wdjordjevic1r,6,2
3,sbignell19,6,3
4,ppointon2l,7,4


In [70]:
user_actors_df.to_csv("../data/user_actors.csv", index=False)
user_genres_df.to_csv("../data/user_genre.csv", index=False)

In [71]:
user_genres_df.loc[user_genres_df['username']=='mmeka']

,username,genre_id,rating
490,mmeka,8,3
541,mmeka,6,4
644,mmeka,15,1
726,mmeka,15,4
842,mmeka,5,4
871,mmeka,1,4
995,mmeka,13,2


In [72]:
user_actors_df.loc[user_actors_df['username']=='mmeka']

,username,actor_id,rating
490,mmeka,737,1
541,mmeka,2542,4
644,mmeka,471,1
726,mmeka,5683,2
842,mmeka,1169,2
871,mmeka,621,3
995,mmeka,5167,1
